# Ingenieria de Caracteristicas

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

In [2]:
INTERIM_DIR = Path("../data/interim").resolve()
RAW_DIR = Path("../data/raw").resolve()

A11Y_ROOT = (
    RAW_DIR
    / "02"
    / "Reduced-A11y-CUA"
)

instances = pd.read_csv(
    INTERIM_DIR / "a11y_instances.csv"
)

behavior = pd.read_csv(
    INTERIM_DIR / "a11y_behavior_features.csv"
)

trees = pd.read_csv(
    INTERIM_DIR / "a11y_tree_session_features.csv"
)

merged = pd.read_csv(
    INTERIM_DIR / "a11y_features_merged.csv"
)

In [3]:
html_files = list(
    A11Y_ROOT.rglob("*.html")
)

print(
    "HTML encontrados:",
    len(html_files)
)

HTML encontrados: 857


In [4]:
def obtener_session_dir(
    archivo,
    raiz
):

    if archivo.parent.name == "web_logs":

        carpeta_sesion = (
            archivo.parent.parent
        )

    else:

        carpeta_sesion = (
            archivo.parent
        )

    return str(
        carpeta_sesion.relative_to(
            raiz
        )
    )

In [5]:
html_sessions = {
    obtener_session_dir(
        archivo,
        A11Y_ROOT
    )
    for archivo in html_files
}

len(html_sessions)

353

In [6]:
coverage = instances[
    [
        "session_dir",
        "instance_id",
        "source_group",
        "source_type"
    ]
].copy()

coverage["has_behavior"] = (
    coverage["session_dir"]
    .isin(
        behavior["session_dir"]
    )
)

coverage["has_a11y_tree"] = (
    coverage["session_dir"]
    .isin(
        trees["session_dir"]
    )
)

coverage["has_html"] = (
    coverage["session_dir"]
    .isin(
        html_sessions
    )
)

In [7]:
coverage.head()

,session_dir,instance_id,source_group,source_type,has_behavior,has_a11y_tree,has_html
0,Qwen3-VL-32B-Instruct/Screen Reader/59,Qwen3_VL_32B_Instruct_Screen_Reader_59,Qwen3-VL-32B-Instruct,agent,False,True,False
1,Qwen3-VL-32B-Instruct/Screen Reader/50,Qwen3_VL_32B_Instruct_Screen_Reader_50,Qwen3-VL-32B-Instruct,agent,False,False,False
2,Qwen3-VL-32B-Instruct/Screen Reader/57,Qwen3_VL_32B_Instruct_Screen_Reader_57,Qwen3-VL-32B-Instruct,agent,False,False,False
3,Qwen3-VL-32B-Instruct/Screen Reader/32,Qwen3_VL_32B_Instruct_Screen_Reader_32,Qwen3-VL-32B-Instruct,agent,False,True,False
4,Qwen3-VL-32B-Instruct/Screen Reader/35,Qwen3_VL_32B_Instruct_Screen_Reader_35,Qwen3-VL-32B-Instruct,agent,False,False,False


In [8]:
coverage[
    [
        "has_behavior",
        "has_a11y_tree",
        "has_html"
    ]
].sum()

has_behavior      353
has_a11y_tree    1262
has_html          353
dtype: int64

In [9]:
pd.crosstab(
    coverage["source_group"],
    [
        coverage["has_behavior"],
        coverage["has_html"]
    ]
)

has_behavior          False       True       
has_html              False True  False True 
source_group                                 
BLVU                    315     1     2   162
Claude-Sonnet-4.5       140     0     0    40
Qwen3-VL-32B-Instruct   174     0     1     5
SU                      335     2     0   143

In [10]:
def tiene_nombre_control(
    elemento,
    soup
):

    # Texto visible
    texto = elemento.get_text(
        " ",
        strip=True
    )

    if texto:
        return True

    # aria-label
    aria = elemento.get(
        "aria-label"
    )

    if aria and aria.strip():
        return True

    # title
    title = elemento.get(
        "title"
    )

    if title and title.strip():
        return True

    # aria-labelledby
    labelledby = elemento.get(
        "aria-labelledby"
    )

    if labelledby:

        for identificador in (
            labelledby.split()
        ):

            etiqueta = soup.find(
                id=identificador
            )

            if (
                etiqueta
                and etiqueta.get_text(
                    " ",
                    strip=True
                )
            ):
                return True

    # <label for="...">
    element_id = elemento.get(
        "id"
    )

    if element_id:

        label = soup.find(
            "label",
            attrs={
                "for": element_id
            }
        )

        if (
            label
            and label.get_text(
                " ",
                strip=True
            )
        ):
            return True

    # <label><input ...></label>
    if elemento.find_parent(
        "label"
    ):
        return True

    return False

In [11]:
def extraer_html_features(
    archivo,
    raiz
):

    with open(
        archivo,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        html = f.read()

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    elementos = soup.find_all(True)

    # --------------------------------
    # CONTROLES INTERACTIVOS
    # --------------------------------

    controles = []

    controles.extend(
        soup.find_all("button")
    )

    controles.extend(
        soup.find_all(
            "a",
            href=True
        )
    )

    controles.extend(
        [
            x
            for x in soup.find_all(
                "input"
            )
            if x.get("type") != "hidden"
        ]
    )

    controles.extend(
        soup.find_all("select")
    )

    controles.extend(
        soup.find_all("textarea")
    )

    # Eliminar duplicados
    controles = list({
        id(x): x
        for x in controles
    }.values())

    controles_con_nombre = sum(
        tiene_nombre_control(
            control,
            soup
        )
        for control in controles
    )

    controles_sin_nombre = (
        len(controles)
        - controles_con_nombre
    )

    if len(controles) > 0:

        label_ratio = (
            controles_con_nombre
            / len(controles)
        )

    else:

        label_ratio = np.nan

    # --------------------------------
    # VALIDACIÓN
    # --------------------------------

    campos = (
        soup.find_all("input")
        + soup.find_all("select")
        + soup.find_all("textarea")
    )

    atributos_validacion = {
        "required",
        "pattern",
        "min",
        "max",
        "minlength",
        "maxlength"
    }

    campos_con_validacion = 0

    for campo in campos:

        if any(
            campo.has_attr(attr)
            for attr
            in atributos_validacion
        ):
            campos_con_validacion += 1

    # --------------------------------
    # PROGRESO
    # --------------------------------

    progress_present = int(
        soup.find("progress")
        is not None

        or

        soup.find(
            attrs={
                "role": "progressbar"
            }
        )
        is not None
    )

    # --------------------------------
    # REGIONES DE ESTADO
    # --------------------------------

    status_present = int(

        soup.find(
            attrs={
                "role": "status"
            }
        )
        is not None

        or

        soup.find(
            attrs={
                "aria-live": True
            }
        )
        is not None

        or

        soup.find(
            attrs={
                "role": "alert"
            }
        )
        is not None
    )

    # --------------------------------
    # BREADCRUMBS
    # --------------------------------

    breadcrumb_present = int(

        soup.find(
            attrs={
                "aria-label":
                    lambda x:
                    x
                    and
                    "breadcrumb"
                    in x.lower()
            }
        )
        is not None

        or

        soup.find(
            class_=
                lambda x:
                x
                and
                "breadcrumb"
                in str(x).lower()
        )
        is not None
    )

    # --------------------------------
    # DOM DEPTH
    # --------------------------------

    depths = []

    for element in elementos:

        depth = len(
            list(element.parents)
        )

        depths.append(depth)

    max_dom_depth = (
        max(depths)
        if depths
        else 0
    )

    return {

        "session_dir":
            obtener_session_dir(
                archivo,
                raiz
            ),

        "source_file":
            str(
                archivo.relative_to(
                    raiz
                )
            ),

        "html_element_count":
            len(elementos),

        "form_count":
            len(
                soup.find_all("form")
            ),

        "input_count":
            len(
                soup.find_all("input")
            ),

        "button_count":
            len(
                soup.find_all("button")
            ),

        "link_count":
            len(
                soup.find_all(
                    "a",
                    href=True
                )
            ),

        "nav_count":
            len(
                soup.find_all("nav")
            ),

        "interactive_controls_count":
            len(controles),

        "labeled_controls_count":
            controles_con_nombre,

        "unlabeled_controls_html_count":
            controles_sin_nombre,

        "control_label_ratio_html":
            label_ratio,

        "validated_fields_count":
            campos_con_validacion,

        "input_validation_present":
            int(
                campos_con_validacion > 0
            ),

        "progress_indicator_present":
            progress_present,

        "status_region_present":
            status_present,

        "breadcrumbs_present":
            breadcrumb_present,

        "max_dom_depth":
            max_dom_depth
    }

In [12]:
html_records = []
html_errors = []

for i, archivo in enumerate(
    html_files,
    start=1
):

    try:

        features = (
            extraer_html_features(
                archivo,
                A11Y_ROOT
            )
        )

        html_records.append(
            features
        )

    except Exception as e:

        html_errors.append({

            "archivo":
                str(archivo),

            "error":
                str(e)
        })

    if i % 100 == 0:

        print(
            f"{i}/{len(html_files)}"
        )

100/857
200/857
300/857
400/857
500/857
600/857
700/857
800/857


In [13]:
html_features = pd.DataFrame(
    html_records
)

print(
    html_features.shape
)

display(
    html_features.head()
)

(857, 18)


,session_dir,source_file,html_element_count,form_count,input_count,button_count,link_count,nav_count,interactive_controls_count,labeled_controls_count,unlabeled_controls_html_count,control_label_ratio_html,validated_fields_count,input_validation_present,progress_indicator_present,status_region_present,breadcrumbs_present,max_dom_depth
0,Qwen3-VL-32B-Instruct/Default/3,Qwen3-VL-32B-Instruct/Default/3/web_logs/web_t...,1730,1,2,29,337,2,367,358,9,0.975477,0,0,0,1,0,20
1,Qwen3-VL-32B-Instruct/Default/1,Qwen3-VL-32B-Instruct/Default/1/web_logs/web_t...,4252,2,10,13,170,0,187,171,16,0.914439,1,1,1,1,0,55
2,Qwen3-VL-32B-Instruct/Default/58,Qwen3-VL-32B-Instruct/Default/58/web_logs/web_...,4713,1,3,147,118,0,268,220,48,0.820896,0,0,0,1,0,32
3,Qwen3-VL-32B-Instruct/Default/58,Qwen3-VL-32B-Instruct/Default/58/web_logs/web_...,1548,1,2,42,3,0,47,30,17,0.638298,0,0,0,1,0,16
4,Qwen3-VL-32B-Instruct/Default/58,Qwen3-VL-32B-Instruct/Default/58/web_logs/web_...,598,1,8,4,4,0,11,10,1,0.909091,0,0,1,1,0,24


In [14]:
html_features.describe().T

,count,mean,std,min,25%,50%,75%,max
html_element_count,857.0,2659.393232,2598.760760,4.0,897.000000,1935.000000,3686.000000,17254.0
form_count,857.0,1.513419,1.499258,0.0,1.000000,1.000000,2.000000,31.0
input_count,857.0,7.880980,13.907482,0.0,2.000000,5.000000,11.000000,251.0
button_count,857.0,28.287048,42.641044,0.0,1.000000,5.000000,41.000000,299.0
link_count,857.0,178.555426,411.412233,0.0,44.000000,100.000000,170.000000,4714.0
nav_count,857.0,0.983664,2.197538,0.0,0.000000,0.000000,1.000000,12.0
interactive_controls_count,857.0,211.890315,417.117103,0.0,57.000000,126.000000,238.000000,4773.0
labeled_controls_count,857.0,200.870478,413.408720,0.0,54.000000,117.000000,211.000000,4729.0
unlabeled_controls_html_count,857.0,11.019837,21.441854,0.0,1.000000,3.000000,12.000000,353.0
control_label_ratio_html,838.0,0.941408,0.080590,0.0,0.919815,0.975477,0.985989,1.0


In [15]:
len(html_errors)

0

In [16]:
html_features.to_csv(
    INTERIM_DIR
    / "a11y_html_features.csv",
    index=False
)

In [17]:
html_session_features = (
    html_features
    .groupby(
        "session_dir",
        as_index=False
    )
    .agg(

        html_snapshot_count=(
            "source_file",
            "count"
        ),

        mean_html_elements=(
            "html_element_count",
            "mean"
        ),

        max_html_elements=(
            "html_element_count",
            "max"
        ),

        mean_interactive_controls_html=(
            "interactive_controls_count",
            "mean"
        ),

        total_interactive_controls_html=(
            "interactive_controls_count",
            "sum"
        ),

        total_labeled_controls_html=(
            "labeled_controls_count",
            "sum"
        ),

        max_unlabeled_controls_html=(
            "unlabeled_controls_html_count",
            "max"
        ),

        input_validation_present=(
            "input_validation_present",
            "max"
        ),

        progress_indicator_present=(
            "progress_indicator_present",
            "max"
        ),

        status_region_present=(
            "status_region_present",
            "max"
        ),

        breadcrumbs_present=(
            "breadcrumbs_present",
            "max"
        ),

        max_dom_depth=(
            "max_dom_depth",
            "max"
        )
    )
)

In [18]:
html_session_features[
    "control_label_ratio_html"
] = np.where(

    html_session_features[
        "total_interactive_controls_html"
    ] > 0,

    html_session_features[
        "total_labeled_controls_html"
    ]
    /
    html_session_features[
        "total_interactive_controls_html"
    ],

    np.nan
)

In [19]:
html_session_features.head()

,session_dir,html_snapshot_count,mean_html_elements,max_html_elements,mean_interactive_controls_html,total_interactive_controls_html,total_labeled_controls_html,max_unlabeled_controls_html,input_validation_present,progress_indicator_present,status_region_present,breadcrumbs_present,max_dom_depth,control_label_ratio_html
0,BLVU/BLVU1/1,3,2952.0,4535,155.333333,466,412,51,1,1,1,0,51,0.884120
1,BLVU/BLVU1/10,1,159.0,159,47.000000,47,46,1,0,0,0,0,19,0.978723
2,BLVU/BLVU1/11,5,826.6,2051,66.600000,333,322,4,1,0,1,1,37,0.966967
3,BLVU/BLVU1/12,2,4372.5,6509,153.000000,306,294,10,1,1,1,0,55,0.960784
4,BLVU/BLVU1/2,1,1192.0,1192,66.000000,66,64,2,1,1,1,0,29,0.969697


In [20]:
html_session_features.to_csv(
    INTERIM_DIR
    / "a11y_html_session_features.csv",
    index=False
)

In [21]:
a11y_features = pd.read_csv(
    INTERIM_DIR
    / "a11y_features_merged.csv"
)

In [22]:
a11y_features_enriched = (
    a11y_features
    .merge(
        html_session_features,
        on="session_dir",
        how="left",
        validate="one_to_one"
    )
)

In [23]:
a11y_features_enriched.shape

(1320, 47)

In [24]:
assert (
    len(a11y_features_enriched)
    ==
    len(a11y_features)
)

In [25]:
a11y_features_enriched[
    "has_behavior_data"
] = (
    a11y_features_enriched[
        "event_count"
    ]
    .notna()
    .astype(int)
)

a11y_features_enriched[
    "has_tree_data"
] = (
    a11y_features_enriched[
        "a11y_snapshot_count"
    ]
    .notna()
    .astype(int)
)

a11y_features_enriched[
    "has_html_data"
] = (
    a11y_features_enriched[
        "html_snapshot_count"
    ]
    .notna()
    .astype(int)
)

In [26]:
a11y_features_enriched[
    [
        "has_behavior_data",
        "has_tree_data",
        "has_html_data"
    ]
].sum()

has_behavior_data     353
has_tree_data        1262
has_html_data         353
dtype: int64

In [27]:
a11y_features_enriched[
    [
        "has_behavior_data",
        "has_tree_data",
        "has_html_data"
    ]
].fillna(0)

,has_behavior_data,has_tree_data,has_html_data
0,0,1,0
1,0,0,0
2,0,0,0
3,0,1,0
4,0,0,0
...,...,...,...
1315,0,1,0
1316,1,1,1
1317,0,1,0
1318,0,1,0


In [28]:
pd.crosstab(
    a11y_features_enriched[
        "source_type"
    ],
    [
        a11y_features_enriched[
            "has_behavior_data"
        ],
        a11y_features_enriched[
            "has_html_data"
        ]
    ]
)

has_behavior_data    0     1     
has_html_data        0  1  0    1
source_type                      
agent              314  0  1   45
human_candidate    650  3  2  305

In [29]:
coverage_summary = (
    a11y_features_enriched
    .groupby(
        "source_type"
    )
    [
        [
            "has_behavior_data",
            "has_tree_data",
            "has_html_data"
        ]
    ]
    .mean()
    * 100
)

coverage_summary

,has_behavior_data,has_tree_data,has_html_data
source_type,,,
agent,12.777778,85.000000,12.500000
human_candidate,31.979167,99.583333,32.083333


In [30]:
a11y_features_enriched.to_csv(
    INTERIM_DIR
    / "a11y_features_enriched.csv",
    index=False
)